# Lab 01 - EDA (FashionMNIST)

Notebook nay thuc thi phan **Exploratory Data Analysis** theo `docs/EDA1.md`, pham vi:

- **Core (bat buoc):** Step 1 structural, Step 2 integrity, Step 3 class distribution,
  Step 4 visual inspection, Step 5 pixel intensity statistics, Step 13 findings.
- **Extended (thuc hien):** Step 6 spatial analysis, Step 8 inter-class similarity.
- **Extended (khong thuc hien):** Step 7 handcrafted features, Step 9 PCA/t-SNE,
  Step 10 outlier scoring, Step 11 distribution shift, Step 12 traditional baseline.
  Ly do ghi ro o phan cuoi notebook va trong `walkthrough_lab_01.md`.

**Hai quy tac duoc thuc thi bang code, khong chi bang ky luat:**

1. Tap test chinh thuc chi duoc dung cho kiem tra cau truc va trung lap noi dung.
   Moi thong ke co the anh huong den quyet dinh mo hinh chi tinh tren train/val.
2. Pixel 0 la nen, khong phai missing value. Foreground duoc dinh nghia bang mot
   nguong tuong minh (`FOREGROUND_THRESHOLD = 0.1`), khong phai `pixel != 0`.

**Notebook nay tao ra artifact ma Feature Engineering va ca ba run se tai su dung:**

- `outputs/eda/split_indices.npz` - chi so train/val, tao MOT lan duy nhat.
- `outputs/eda/split_metadata.json` - provenance cua split.
- `outputs/eda/tables/*.csv` - cac bang thong ke.
- `outputs/eda/figures/*.png` - cac bieu do.

## E0. Thiet lap moi truong va import

Notebook tu nhan dien Local va Kaggle giong notebook training, de chay lai duoc o ca hai noi.
Khong doc du lieu qua `get_dataloaders()`: ham do tra ve tensor da Normalize, sai view cho EDA
(khong the kiem tra pixel range hay dtype goc tren tensor da chuan hoa).

In [1]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # rendering khong tuong tac: notebook chay duoc bang nbconvert
import matplotlib.pyplot as plt

# 1. Thiet lap PROJECT_ROOT giong notebook training
if os.path.exists("/kaggle/working"):
    sys.path.insert(0, "/kaggle/working")
    PROJECT_ROOT = Path("/kaggle/working")
else:
    PROJECT_ROOT = Path("..").resolve()
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    DATA_DIR, EDA_DIR, OUTPUTS_DIR, CLASS_NAMES, RANDOM_SEED,
    VAL_SPLIT, SPLIT_VERSION, NUM_CLASSES,
)
from src.split import make_stratified_split, save_split, load_split_metadata
from src import eda as eda_mod
from src.eda import FOREGROUND_THRESHOLD
from src.utils import set_seed

TABLES_DIR = EDA_DIR / "tables"
FIGURES_DIR = EDA_DIR / "figures"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

set_seed(RANDOM_SEED)

print(f"[PROJECT_ROOT] {PROJECT_ROOT}")
print(f"[DATA_DIR]     {DATA_DIR}")
print(f"[EDA_DIR]      {EDA_DIR}")
print(f"[SPLIT_VERSION] {SPLIT_VERSION}")
print(f"[FOREGROUND_THRESHOLD] {FOREGROUND_THRESHOLD}")
print(f"[CLASS_NAMES] {len(CLASS_NAMES)} classes")

[PROJECT_ROOT] C:\Users\Win 11\Documents\Lab DL\labs\lab01
[DATA_DIR]     C:\Users\Win 11\Documents\Lab DL\labs\lab01\data
[EDA_DIR]      C:\Users\Win 11\Documents\Lab DL\labs\lab01\outputs\eda
[SPLIT_VERSION] split_v2_stratified
[FOREGROUND_THRESHOLD] 0.1
[CLASS_NAMES] 10 classes


## E1. Nap du lieu raw va tao split mot lan duy nhat

Doc truc tiep file IDX de giu nguyen `uint8` va khoang `[0, 255]`.

**Vi sao phai stratified:** `torch.utils.data.random_split` khong bao toan ty le lop.
Voi FashionMNIST, mot lan chia seed=42 cho ra validation tu 537 den 658 anh moi lop
(thay vi 600) - lech toi +/-10%. Split lech lam cho val accuracy khong con la thuoc do
cong bang giua cac kien truc, va lam cho thong ke normalization do tren "train" phu thuoc
vao mot tap con khong dai dien.

Split duoc luu ra artifact ngay tai day. EDA, Feature Engineering va ca ba run deu doc lai
cung mot artifact nay; khong noi nao chia lai.

In [2]:
X_train_pool, y_train_pool, X_test_raw, y_test_raw = eda_mod.load_raw_arrays(DATA_DIR)

print(f"[RAW] train pool : X={X_train_pool.shape} {X_train_pool.dtype} | y={y_train_pool.shape} {y_train_pool.dtype}")
print(f"[RAW] test       : X={X_test_raw.shape} {X_test_raw.dtype} | y={y_test_raw.shape} {y_test_raw.dtype}")
print(f"[RAW] pixel range: [{X_train_pool.min()}, {X_train_pool.max()}] (train) | "
      f"[{X_test_raw.min()}, {X_test_raw.max()}] (test)")

# Tao split MOT lan. Moi stage sau deu doc lai artifact nay.
train_idx, val_idx = make_stratified_split(
    y_train_pool, val_split=VAL_SPLIT, seed=RANDOM_SEED
)

npz_path = save_split(
    out_dir=EDA_DIR,
    train_idx=train_idx,
    val_idx=val_idx,
    targets=y_train_pool,
    seed=RANDOM_SEED,
    val_split=VAL_SPLIT,
    version=SPLIT_VERSION,
    extra={
        "library_versions": {
            "numpy": np.__version__,
            "pandas": pd.__version__,
        },
        "foreground_threshold": FOREGROUND_THRESHOLD,
    },
)

split_meta = load_split_metadata(EDA_DIR)
print(f"\n[SPLIT] artifact: {npz_path}")
print(f"[SPLIT] version={split_meta['split_version']} seed={split_meta['seed']} "
      f"val_split={split_meta['val_split']} stratified={split_meta['stratified']}")
print(f"[SPLIT] pool={split_meta['pool_size']} train={split_meta['train_size']} val={split_meta['val_size']}")
print(f"[SPLIT] train per-class: {split_meta['train_class_counts']}")
print(f"[SPLIT] val   per-class: {split_meta['val_class_counts']}")
print(f"[SPLIT] excluded: {split_meta['excluded_partition']}")

# Kiem tra chen cheo va coverage - neu that bai thi dung ngay, khong di tiep
assert np.intersect1d(train_idx, val_idx).size == 0, "train/val indices overlap"
assert len(train_idx) + len(val_idx) == len(y_train_pool), "split does not cover the pool"
assert min(split_meta["val_class_counts"]) == max(split_meta["val_class_counts"]), \
    "validation partition is not class-balanced"

X_tr, y_tr = X_train_pool[train_idx], y_train_pool[train_idx]
X_va, y_va = X_train_pool[val_idx], y_train_pool[val_idx]
print(f"\n[OK] train {X_tr.shape} | val {X_va.shape} | test {X_test_raw.shape} (held out)")

[RAW] train pool : X=(60000, 28, 28) uint8 | y=(60000,) uint8
[RAW] test       : X=(10000, 28, 28) uint8 | y=(10000,) uint8
[RAW] pixel range: [0, 255] (train) | [0, 255] (test)

[SPLIT] artifact: C:\Users\Win 11\Documents\Lab DL\labs\lab01\outputs\eda\split_indices.npz
[SPLIT] version=split_v2_stratified seed=42 val_split=0.1 stratified=True
[SPLIT] pool=60000 train=54000 val=6000
[SPLIT] train per-class: [5400, 5400, 5400, 5400, 5400, 5400, 5400, 5400, 5400, 5400]
[SPLIT] val   per-class: [600, 600, 600, 600, 600, 600, 600, 600, 600, 600]
[SPLIT] excluded: FashionMNIST train=False (10,000 official test images)

[OK] train (54000, 28, 28) | val (6000, 28, 28) | test (10000, 28, 28) (held out)


## E2. Step 1 - Structural overview

Muc tieu: xac lap cau truc co ban (shape, dtype, pixel range, memory) bang so lieu do duoc,
khong phai bang mo ta. Bang nay la bang chung cho phan "Dataset overview" cua bao cao.

In [3]:
overview = eda_mod.structural_overview(
    X_train_pool, y_train_pool, X_test_raw, y_test_raw, CLASS_NAMES
)
overview.to_csv(TABLES_DIR / "structural_overview.csv", index=False)

print(overview.to_string(index=False))

     partition  num_images  image_height  image_width  num_channels     array_shape dtype  pixel_min  pixel_max  num_labels  label_min  label_max  num_classes_present  num_class_names_defined  memory_MB
original_train       60000            28           28             1 (60000, 28, 28) uint8          0        255       60000          0          9                   10                       10     44.861
 original_test       10000            28           28             1 (10000, 28, 28) uint8          0        255       10000          0          9                   10                       10      7.477
       summary       70000            28           28             1             n/a uint8          0        255       70000          0          9                   10                       10     52.338


## E3. Step 2 - Data integrity check

Sau nhom kiem tra, phan biet ro hai khai niem thuong bi gop lam mot:

- **Index disjointness:** split khong dung lai cung mot dong hai lan.
- **Content duplication:** hai dong khac nhau chua pixel giong het nhau.

Mot dataset co the pass kiem tra thu nhat va van fail kiem tra thu hai. Ca hai deu duoc do.

Khong xoa anh nao. Cac anh nghi van chi duoc liet ke kem `source_dataset`,
`original_index`, `split`, `label` de truy nguoc.

In [4]:
integrity_summary, integrity_details = eda_mod.integrity_report(
    X_train=X_train_pool,
    y_train=y_train_pool,
    X_test=X_test_raw,
    y_test=y_test_raw,
    train_indices=train_idx,
    val_indices=val_idx,
    num_classes=NUM_CLASSES,
)

integrity_summary.to_csv(TABLES_DIR / "integrity_report.csv", index=False)
print(integrity_summary.to_string(index=False))

print("\n[CHI TIET CAC MAU NGHI VAN]")
for name, frame in integrity_details.items():
    print(f"  {name:<32} rows={len(frame)}")
    if len(frame):
        frame.to_csv(TABLES_DIR / f"integrity_{name}.csv", index=False)

# Tong hop trung lap theo split de thay ro duplicate nam trong train hay val
dup_train = integrity_details["exact_duplicate_train"]
if len(dup_train):
    print("\n[DUPLICATE THEO SPLIT]")
    print(dup_train["split"].value_counts().to_string())
else:
    print("\n[DUPLICATE THEO SPLIT] khong co exact duplicate trong train pool")

                        check                                                    expected  num_flagged                               detail_file
              labels_in_range                                                      [0, 9]            0 invalid_train_labels, invalid_test_labels
            pixel_range_0_255                                  all pixels within [0, 255]            0                                       n/a
            nan_or_inf_pixels                  0 (uint8 storage cannot represent NaN/Inf)            0                                       n/a
             all_black_images                               0 images with maximum pixel 0            0           all_black_train, all_black_test
       split_indices_disjoint                                       0 overlapping indices            0                                       n/a
               split_coverage                          60000 samples covered exactly once            0                            

## E4. Step 3 - Class distribution

Spec cam mac dinh rang phan phoi can bang, nen bang duoc tinh tu du lieu thuc te.
Tap test khong nam trong bang nay: no chi duoc kiem tra tong so mau o buoc cuoi.

In [5]:
distribution = eda_mod.class_distribution(y_tr, y_va, CLASS_NAMES)
distribution.to_csv(TABLES_DIR / "class_distribution.csv", index=False)

print(distribution.to_string(index=False))

train_counts = np.bincount(y_tr, minlength=NUM_CLASSES)
val_counts = np.bincount(y_va, minlength=NUM_CLASSES)
print(f"\n[TRAIN] min={train_counts.min()} max={train_counts.max()} "
      f"ratio_max_min={train_counts.max() / train_counts.min():.4f}")
print(f"[VAL]   min={val_counts.min()} max={val_counts.max()} "
      f"ratio_max_min={val_counts.max() / val_counts.min():.4f}")
print(f"[TEST]  held out, counts withheld until the configuration is locked")

eda_mod.plot_class_distribution(distribution, save_path=FIGURES_DIR / "class_distribution.png")
plt.close("all")
print(f"\n[FIG] {FIGURES_DIR / 'class_distribution.png'}")

split  label  class_name  count  percent  split_total
train      0 T-shirt/top   5400     10.0        54000
train      1     Trouser   5400     10.0        54000
train      2    Pullover   5400     10.0        54000
train      3       Dress   5400     10.0        54000
train      4        Coat   5400     10.0        54000
train      5      Sandal   5400     10.0        54000
train      6       Shirt   5400     10.0        54000
train      7     Sneaker   5400     10.0        54000
train      8         Bag   5400     10.0        54000
train      9  Ankle boot   5400     10.0        54000
  val      0 T-shirt/top    600     10.0         6000
  val      1     Trouser    600     10.0         6000
  val      2    Pullover    600     10.0         6000
  val      3       Dress    600     10.0         6000
  val      4        Coat    600     10.0         6000
  val      5      Sandal    600     10.0         6000
  val      6       Shirt    600     10.0         6000
  val      7     Sneaker    


[FIG] C:\Users\Win 11\Documents\Lab DL\labs\lab01\outputs\eda\figures\class_distribution.png


## E5. Step 4 - Visual inspection

Lay mau ngau nhien theo seed co dinh, **khong lay N anh dau tien**: du lieu IDX duoc sap xep,
nen phan dau file khong dai dien cho toan bo phan phoi.

Mean image theo lop duoc tinh them de thay "hinh dang trung binh" cua tung lop. Luu y:
mean image lam mat phuong sai trong lop - chinh la thu quyet dinh viec hai lop co de nham
lan hay khong - nen no chi dung de tao gia thuyet, khong dung de ket luan.

In [6]:
N_PER_CLASS = 20
picks = eda_mod.sample_images_per_class(
    X_tr, y_tr, CLASS_NAMES, n_per_class=N_PER_CLASS, seed=RANDOM_SEED
)
print(f"[SAMPLE] {N_PER_CLASS} anh x {len(CLASS_NAMES)} lop = {picks.size} anh, seed={RANDOM_SEED}")

eda_mod.plot_sample_grid(
    X_tr, picks, y_tr, CLASS_NAMES, save_path=FIGURES_DIR / "sample_images.png"
)
plt.close("all")

mean_images = eda_mod.class_mean_images(X_tr, y_tr, CLASS_NAMES)
eda_mod.plot_class_mean_images(
    mean_images, CLASS_NAMES, save_path=FIGURES_DIR / "class_mean_images.png"
)
plt.close("all")

print(f"[FIG] {FIGURES_DIR / 'sample_images.png'}")
print(f"[FIG] {FIGURES_DIR / 'class_mean_images.png'}")

# Luu mean image duoi dang bang de Step 8 dung lai, khong tinh lai
np.save(EDA_DIR / "class_mean_images.npy", mean_images)
print(f"[ARTIFACT] {EDA_DIR / 'class_mean_images.npy'} shape={mean_images.shape}")

[SAMPLE] 20 anh x 10 lop = 200 anh, seed=42


[FIG] C:\Users\Win 11\Documents\Lab DL\labs\lab01\outputs\eda\figures\sample_images.png
[FIG] C:\Users\Win 11\Documents\Lab DL\labs\lab01\outputs\eda\figures\class_mean_images.png
[ARTIFACT] C:\Users\Win 11\Documents\Lab DL\labs\lab01\outputs\eda\class_mean_images.npy shape=(10, 28, 28)


## E6. Step 5 - Pixel intensity statistics

Day la buoc quyet dinh cho normalization o Feature Engineering.

**Quy tac (tu `docs/FEATURE-ENGINEERING.md` Step 3):** mean/std phai la thong ke gop tren
toan bo pixel cua train, dung population variance (`ddof=0`). Khong duoc lay trung binh
cua std tung anh.

Hai gia tri nay khac nhau dang ke tren FashionMNIST, nen ham `compute_normalization_stats`
tra ve ca `per_image_std_mean` nhu mot chan doan, de su khac biet khong bi che khuat.

In [7]:
train_stats = eda_mod.compute_normalization_stats(X_tr)
val_stats = eda_mod.compute_normalization_stats(X_va)

print("[TRAIN PARTITION - scaled [0,1], ddof=0]")
for k, v in train_stats.items():
    print(f"  {k:<26} {v}")
print("\n[VAL PARTITION - descriptive only, never used to fit Normalize]")
for k in ("mean", "std", "num_images", "zero_pixel_fraction"):
    print(f"  {k:<26} {val_stats[k]}")

print("\n[SO SANH VOI CONFIG CU]")
from src.config import FALLBACK_MEAN, FALLBACK_STD
print(f"  legacy mean {FALLBACK_MEAN[0]:.6f} vs measured {train_stats['mean']:.6f} "
      f"(delta {train_stats['mean'] - FALLBACK_MEAN[0]:+.6f})")
print(f"  legacy std  {FALLBACK_STD[0]:.6f} vs measured {train_stats['std']:.6f} "
      f"(delta {train_stats['std'] - FALLBACK_STD[0]:+.6f})")
print(f"  legacy std vs mean-of-per-image-std {train_stats['per_image_std_mean']:.6f} "
      f"(delta {FALLBACK_STD[0] - train_stats['per_image_std_mean']:+.6f})")
print("  -> legacy std khop voi mean-of-per-image-std, KHONG phai global std.")

[TRAIN PARTITION - scaled [0,1], ddof=0]
  mean                       0.28620901703834534
  std                        0.35315975546836853
  num_images                 54000
  num_pixels                 42336000
  per_image_std_mean         0.3206605911254883
  per_image_std_ddof0_mean   0.32045602798461914
  per_image_mean_mean        0.28620901703834534
  zero_pixel_fraction        0.5021429988662132

[VAL PARTITION - descriptive only, never used to fit Normalize]
  mean                       0.28452491760253906
  std                        0.35179877281188965
  num_images                 6000
  zero_pixel_fraction        0.5012270408163265

[SO SANH VOI CONFIG CU]
  legacy mean 0.286000 vs measured 0.286209 (delta +0.000209)
  legacy std  0.320500 vs measured 0.353160 (delta +0.032660)
  legacy std vs mean-of-per-image-std 0.320661 (delta -0.000161)
  -> legacy std khop voi mean-of-per-image-std, KHONG phai global std.


In [8]:
pixel_by_class = eda_mod.pixel_statistics_by_class(X_tr, y_tr, CLASS_NAMES)
pixel_by_class.to_csv(TABLES_DIR / "pixel_statistics.csv", index=False)
print(pixel_by_class.to_string(index=False))

stats_table = pd.DataFrame(
    [
        {"partition": "train", **train_stats},
        {"partition": "val", **val_stats},
    ]
)
stats_table.to_csv(TABLES_DIR / "normalization_stats_measurement.csv", index=False)

eda_mod.plot_pixel_histogram(X_tr, X_va, save_path=FIGURES_DIR / "pixel_histogram.png")
plt.close("all")
eda_mod.plot_brightness_distribution(
    X_tr, y_tr, CLASS_NAMES, save_path=FIGURES_DIR / "image_brightness_distribution.png"
)
plt.close("all")

print(f"\n[FIG] {FIGURES_DIR / 'pixel_histogram.png'}")
print(f"[FIG] {FIGURES_DIR / 'image_brightness_distribution.png'}")

 label  class_name  count  pixel_mean  pixel_std  zero_pixel_fraction  image_brightness_mean  image_brightness_std  image_brightness_min  image_brightness_max
     0 T-shirt/top   5400    0.325581   0.350651             0.406169               0.325581              0.113085              0.038250              0.682273
     1     Trouser   5400    0.223160   0.343632             0.650929               0.223159              0.057078              0.059329              0.697059
     2    Pullover   5400    0.377130   0.359160             0.351639               0.377130              0.128223              0.051671              0.740041
     3       Dress   5400    0.258485   0.354167             0.573341               0.258485              0.083648              0.038570              0.559224
     4        Coat   5400    0.384889   0.376076             0.398063               0.384889              0.103778              0.043487              0.698624
     5      Sandal   5400    0.136754   0.2630


[FIG] C:\Users\Win 11\Documents\Lab DL\labs\lab01\outputs\eda\figures\pixel_histogram.png
[FIG] C:\Users\Win 11\Documents\Lab DL\labs\lab01\outputs\eda\figures\image_brightness_distribution.png


## E7. Step 6 - Spatial analysis

Foreground duoc dinh nghia bang nguong `FOREGROUND_THRESHOLD = 0.1` tren thang `[0, 1]`,
khong phai `pixel != 0`: khoang 50% pixel FashionMNIST bang 0, nen quy tac "khac 0" se tinh
ca nhieu hat nhieu gan den vien quan ao la vat the va lam phong moi bounding box.

Ba ban do duoc ve cung nhau vi chung tra loi ba cau hoi khac nhau:

- mean map: vung nao trung binh sang.
- variance map: vung nao thay doi nhieu giua cac anh.
- zero-frequency map: vung nao gan nhu luon la nen.

Mot pixel bien gioi co the bang 0 trong 99% anh va van co phuong sai lon nhat - hai chi so
khong thay the nhau.

In [9]:
spatial = eda_mod.spatial_features(
    X_tr, y_tr, CLASS_NAMES, threshold=FOREGROUND_THRESHOLD
)
# Gan nhan split de moi dong truy nguoc duoc ve partition
spatial["split"] = "train"
spatial.to_csv(TABLES_DIR / "spatial_features.csv", index=False)

print(f"[SPATIAL] {len(spatial)} anh, threshold={FOREGROUND_THRESHOLD}")
print(f"[SPATIAL] empty foreground: {int(spatial['empty_foreground'].sum())} anh")
print("\n[SPATIAL] trung binh theo lop:")
print(
    spatial.groupby("class_name")[
        ["foreground_ratio", "bbox_height", "bbox_width", "bbox_aspect_ratio", "bbox_area_ratio"]
    ].mean().round(4).to_string()
)

mean_map = eda_mod.mean_pixel_map(X_tr)
variance_map = eda_mod.pixel_variance_map(X_tr)
zero_map = eda_mod.zero_pixel_frequency_map(X_tr)
np.savez_compressed(
    EDA_DIR / "spatial_maps.npz",
    mean_map=mean_map, variance_map=variance_map, zero_frequency_map=zero_map,
)

eda_mod.plot_heatmaps(
    mean_map, variance_map, zero_map, save_path=FIGURES_DIR / "spatial_heatmaps.png"
)
plt.close("all")
eda_mod.plot_spatial_by_class(
    spatial, CLASS_NAMES, save_path=FIGURES_DIR / "spatial_by_class.png"
)
plt.close("all")

print(f"\n[FIG] {FIGURES_DIR / 'spatial_heatmaps.png'}")
print(f"[FIG] {FIGURES_DIR / 'spatial_by_class.png'}")
print(f"[ARTIFACT] {EDA_DIR / 'spatial_maps.npz'}")

[SPATIAL] 54000 anh, threshold=0.1
[SPATIAL] empty foreground: 0 anh

[SPATIAL] trung binh theo lop:
             foreground_ratio  bbox_height  bbox_width  bbox_aspect_ratio  bbox_area_ratio
class_name                                                                                
Ankle boot             0.4355      21.1307     27.7283             1.3585           0.7455
Bag                    0.5358      23.3676     25.5565             1.1740           0.7487
Coat                   0.5650      27.8181     20.1198             0.7239           0.7136
Dress                  0.3915      27.8987     14.6376             0.5250           0.5208
Pullover               0.6075      27.7922     21.1994             0.7636           0.7512
Sandal                 0.2526      16.1011     27.8096             2.0323           0.5695
Shirt                  0.5772      27.7065     21.4567             0.7772           0.7571
Sneaker                0.2872      13.4137     27.8548             2.1499       


[FIG] C:\Users\Win 11\Documents\Lab DL\labs\lab01\outputs\eda\figures\spatial_heatmaps.png
[FIG] C:\Users\Win 11\Documents\Lab DL\labs\lab01\outputs\eda\figures\spatial_by_class.png
[ARTIFACT] C:\Users\Win 11\Documents\Lab DL\labs\lab01\outputs\eda\spatial_maps.npz


## E8. Step 8 - Inter-class similarity

Cosine tren mean image cua tung lop. Day la **may tao gia thuyet**, khong phai bang chung
ve do kho phan loai: mean image da lam mat phuong sai trong lop, ma chinh phuong sai do moi
quyet dinh viec hai lop co nham lan hay khong. Ket luan ve cap lop kho chi duoc dua ra sau khi
co confusion matrix cua mo hinh da huan luyen.

Hai metric duoc tinh vi tra loi hai cau hoi hoi khac nhau: cosine so sanh hinh dang profile
sau khi bo do lon, correlation bo them muc sang trung binh cua tung anh.

In [10]:
cosine_sim = eda_mod.class_similarity_matrix(mean_images, CLASS_NAMES, metric="cosine")
corr_sim = eda_mod.class_similarity_matrix(mean_images, CLASS_NAMES, metric="correlation")

cosine_sim.to_csv(TABLES_DIR / "class_similarity_matrix.csv")
corr_sim.to_csv(TABLES_DIR / "class_correlation_matrix.csv")

top_cosine = eda_mod.most_similar_pairs(cosine_sim, top_k=12)
top_corr = eda_mod.most_similar_pairs(corr_sim, top_k=12)
top_cosine.to_csv(TABLES_DIR / "most_similar_pairs_cosine.csv", index=False)
top_corr.to_csv(TABLES_DIR / "most_similar_pairs_correlation.csv", index=False)

print("[TOP 12 CAP TUONG DONG NHAT - COSINE]")
print(top_cosine.to_string(index=False))
print("\n[TOP 12 CAP TUONG DONG NHAT - CORRELATION]")
print(top_corr.to_string(index=False))

eda_mod.plot_similarity_heatmap(
    cosine_sim, "cosine", save_path=FIGURES_DIR / "class_similarity_heatmap.png"
)
plt.close("all")
print(f"\n[FIG] {FIGURES_DIR / 'class_similarity_heatmap.png'}")

[TOP 12 CAP TUONG DONG NHAT - COSINE]
    class_a    class_b  similarity
       Coat      Shirt    0.987286
   Pullover       Coat    0.984075
   Pullover      Shirt    0.982976
T-shirt/top      Shirt    0.949727
     Sandal Ankle boot    0.926082
    Trouser      Dress    0.924922
T-shirt/top      Dress    0.923231
     Sandal    Sneaker    0.914833
T-shirt/top       Coat    0.906569
T-shirt/top   Pullover    0.899379
       Coat        Bag    0.893556
      Shirt        Bag    0.893320

[TOP 12 CAP TUONG DONG NHAT - CORRELATION]
    class_a    class_b  similarity
       Coat      Shirt    0.966597
   Pullover       Coat    0.957573
   Pullover      Shirt    0.947404
    Trouser      Dress    0.878530
     Sandal    Sneaker    0.876002
T-shirt/top      Shirt    0.868601
T-shirt/top      Dress    0.858449
     Sandal Ankle boot    0.848523
      Dress      Shirt    0.786858
T-shirt/top       Coat    0.758001
     Sandal        Bag    0.756178
      Dress       Coat    0.739579



[FIG] C:\Users\Win 11\Documents\Lab DL\labs\lab01\outputs\eda\figures\class_similarity_heatmap.png


## E9. Step 13 - Findings, gioi han va ban giao sang Feature Engineering

Phan nay **khong viet ket luan dinh luong san**. Moi con so duoi day duoc doc tu bien vua tinh,
va moi phat bieu duoc gan nhan ro rang: **da do** hay **gia thuyet**.

Bang ban giao sang Feature Engineering theo format cua `docs/EDA1.md`:
`phat hien | split/view | bang chung/artifact | de xuat | cach kiem chung tren validation | trang thai`.

In [11]:
# Gom cac con so DA DO vao mot dict de phan nhan xet chi duoc phep doc tu day.
findings = {
    "split_version": split_meta["split_version"],
    "train_size": int(len(y_tr)),
    "val_size": int(len(y_va)),
    "test_size_held_out": int(len(y_test_raw)),
    "train_class_count_min": int(train_counts.min()),
    "train_class_count_max": int(train_counts.max()),
    "val_class_count_min": int(val_counts.min()),
    "val_class_count_max": int(val_counts.max()),
    "train_pixel_mean": round(train_stats["mean"], 6),
    "train_pixel_std_ddof0": round(train_stats["std"], 6),
    "train_per_image_std_mean": round(train_stats["per_image_std_mean"], 6),
    "legacy_config_std": float(FALLBACK_STD[0]),
    "zero_pixel_fraction_train": round(train_stats["zero_pixel_fraction"], 6),
    "foreground_threshold": FOREGROUND_THRESHOLD,
    "foreground_ratio_mean": round(float(spatial["foreground_ratio"].mean()), 6),
    "empty_foreground_images": int(spatial["empty_foreground"].sum()),
    "exact_duplicate_within_train": int(
        integrity_summary.loc[
            integrity_summary["check"] == "exact_duplicate_within_train", "num_flagged"
        ].iloc[0]
    ),
    "exact_duplicate_train_vs_test": int(
        integrity_summary.loc[
            integrity_summary["check"] == "exact_duplicate_train_vs_test", "num_flagged"
        ].iloc[0]
    ),
    "all_black_images": int(
        integrity_summary.loc[
            integrity_summary["check"] == "all_black_images", "num_flagged"
        ].iloc[0]
    ),
    "top_similar_pair_cosine": f"{top_cosine.iloc[0]['class_a']} / "
                                f"{top_cosine.iloc[0]['class_b']} = "
                                f"{top_cosine.iloc[0]['similarity']:.4f}",
    "top_similar_pair_correlation": f"{top_corr.iloc[0]['class_a']} / "
                                    f"{top_corr.iloc[0]['class_b']} = "
                                    f"{top_corr.iloc[0]['similarity']:.4f}",
}

with open(EDA_DIR / "eda_findings.json", "w", encoding="utf-8") as f:
    json.dump(findings, f, indent=2)

print(json.dumps(findings, indent=2))
print(f"\n[ARTIFACT] {EDA_DIR / 'eda_findings.json'}")

{
  "split_version": "split_v2_stratified",
  "train_size": 54000,
  "val_size": 6000,
  "test_size_held_out": 10000,
  "train_class_count_min": 5400,
  "train_class_count_max": 5400,
  "val_class_count_min": 600,
  "val_class_count_max": 600,
  "train_pixel_mean": 0.286209,
  "train_pixel_std_ddof0": 0.35316,
  "train_per_image_std_mean": 0.320661,
  "legacy_config_std": 0.3205,
  "zero_pixel_fraction_train": 0.502143,
  "foreground_threshold": 0.1,
  "foreground_ratio_mean": 0.451605,
  "empty_foreground_images": 0,
  "exact_duplicate_within_train": 0,
  "exact_duplicate_train_vs_test": 0,
  "all_black_images": 0,
  "top_similar_pair_cosine": "Coat / Shirt = 0.9873",
  "top_similar_pair_correlation": "Coat / Shirt = 0.9666"
}

[ARTIFACT] C:\Users\Win 11\Documents\Lab DL\labs\lab01\outputs\eda\eda_findings.json


In [12]:
# Bang ban giao sang Feature Engineering. Cot "trang thai" chi nhan 3 gia tri:
#   measured  = da do trong notebook nay
#   hypothesis = gia thuyet, phai kiem chung bang confusion matrix sau khi train
#   not_run   = chua thuc hien, co ly do ghi kem
handoff = pd.DataFrame(
    [
        {
            "finding": "Split stratified 54,000/6,000, moi lop dung 5,400/600",
            "split_view": "index-only, raw",
            "evidence": "split_metadata.json, class_distribution.csv",
            "proposal": "Dung dung artifact nay cho EDA, FE va ca 3 run; khong chia lai",
            "validation_check": "assert min==max class count trong moi partition",
            "status": "measured",
        },
        {
            "finding": f"Global train pixel mean={train_stats['mean']:.6f}, "
                       f"std(ddof=0)={train_stats['std']:.6f}",
            "split_view": "train, scaled [0,1], truoc augmentation",
            "evidence": "normalization_stats_measurement.csv",
            "proposal": "FE do lai tren dung artifact nay va ghi preprocessing_config.json",
            "validation_check": "std cua tensor sau Normalize phai ~1.0 (tolerance 0.02)",
            "status": "measured",
        },
        {
            "finding": f"Config cu dung std={FALLBACK_STD[0]:.4f}, bang mean cua per-image std "
                       f"({train_stats['per_image_std_mean']:.6f}), khong phai global std",
            "split_view": "train, scaled [0,1]",
            "evidence": "normalization_stats_measurement.csv (cot per_image_std_mean)",
            "proposal": "Thay bang global std do tren split moi; version hoa preprocessing",
            "validation_check": "test_normalization_yields_unit_variance",
            "status": "measured",
        },
        {
            "finding": f"Khong co exact duplicate trong train pool va khong co "
                       f"train-test overlap ({findings['exact_duplicate_train_vs_test']})",
            "split_view": "raw, md5 tren byte",
            "evidence": "integrity_report.csv",
            "proposal": "Khong loai bo mau nao; giu nguyen dataset goc",
            "validation_check": "hash lai trong FE guard, phai van bang 0",
            "status": "measured",
        },
        {
            "finding": f"Pixel 0 chiem {train_stats['zero_pixel_fraction']:.4f} tong pixel",
            "split_view": "train, scaled [0,1]",
            "evidence": "pixel_statistics.csv, pixel_histogram.png",
            "proposal": "Foreground phai dung nguong, khong dung pixel != 0",
            "validation_check": "foreground_ratio khong doi nghia khi threshold thay doi nho",
            "status": "measured",
        },
        {
            "finding": f"Cap lop giong nhau nhat (cosine): {findings['top_similar_pair_cosine']}",
            "split_view": "train, mean image",
            "evidence": "class_similarity_matrix.csv, class_similarity_heatmap.png",
            "proposal": "Chi ghi la gia thuyet ve cap lop de nham lan",
            "validation_check": "confusion matrix cua ca 3 run phai xac nhan hoac bac bo",
            "status": "hypothesis",
        },
        {
            "finding": "Handcrafted features (Step 7)",
            "split_view": "n/a",
            "evidence": "n/a",
            "proposal": "Khong dua vao model; chi dung de mo ta",
            "validation_check": "n/a",
            "status": "not_run",
        },
        {
            "finding": "PCA / t-SNE (Step 9)",
            "split_view": "n/a",
            "evidence": "n/a",
            "proposal": "Khong thuc hien: khong phuc vu quyet dinh nao cho 3 run MLP/CNN",
            "validation_check": "n/a",
            "status": "not_run",
        },
        {
            "finding": "Outlier scoring (Step 10)",
            "split_view": "n/a",
            "evidence": "n/a",
            "proposal": "Khong thuc hien: spec cam loai bo mau chi vi bi danh dau outlier",
            "validation_check": "n/a",
            "status": "not_run",
        },
        {
            "finding": "Distribution shift train/val (Step 11)",
            "split_view": "n/a",
            "evidence": "pixel_histogram.png (so sanh dinh tinh)",
            "proposal": "Khong chay KS test: hai tap cung mot pool, khac biet chu yeu la sampling noise",
            "validation_check": "n/a",
            "status": "not_run",
        },
        {
            "finding": "Traditional baseline LR/kNN (Step 12)",
            "split_view": "n/a",
            "evidence": "n/a",
            "proposal": "Khong thuc hien: baseline cua lab la BaselineMLP, khong phai mo hinh tuyen tinh",
            "validation_check": "n/a",
            "status": "not_run",
        },
    ]
)
handoff.to_csv(TABLES_DIR / "eda_to_fe_handoff.csv", index=False)
print(handoff.to_string(index=False))

                                                                                 finding                              split_view                                                     evidence                                                                        proposal                                            validation_check     status
                                   Split stratified 54,000/6,000, moi lop dung 5,400/600                         index-only, raw                  split_metadata.json, class_distribution.csv                  Dung dung artifact nay cho EDA, FE va ca 3 run; khong chia lai             assert min==max class count trong moi partition   measured
                                  Global train pixel mean=0.286209, std(ddof=0)=0.353160 train, scaled [0,1], truoc augmentation                          normalization_stats_measurement.csv               FE do lai tren dung artifact nay va ghi preprocessing_config.json     std cua tensor sau Normalize phai ~1.0 (tole

## E10. Tong ket artifact

Liet ke moi file da ghi de nguoi doc kiem tra duoc, va de notebook Feature Engineering biet
chinh xac no duoc phep doc gi.

In [13]:
artifacts = sorted(
    [p for p in EDA_DIR.rglob("*") if p.is_file()],
    key=lambda p: str(p),
)
print(f"[EDA ARTIFACTS] {len(artifacts)} file trong {EDA_DIR}")
for p in artifacts:
    print(f"  {p.relative_to(EDA_DIR)}  ({p.stat().st_size / 1024:.1f} KB)")

required_for_fe = [
    EDA_DIR / "split_indices.npz",
    EDA_DIR / "split_metadata.json",
    EDA_DIR / "eda_findings.json",
]
missing = [str(p) for p in required_for_fe if not p.exists()]
assert not missing, f"Thieu artifact bat buoc cho Feature Engineering: {missing}"
print("\n[OK] Artifact bat buoc cho Feature Engineering da day du:")
for p in required_for_fe:
    print(f"  {p.name}")

[EDA ARTIFACTS] 30 file trong C:\Users\Win 11\Documents\Lab DL\labs\lab01\outputs\eda
  class_mean_images.npy  (30.8 KB)
  eda_findings.json  (0.7 KB)
  feature_engineering_summary.json  (0.4 KB)
  figures\augmentation_preview.png  (61.9 KB)
  figures\class_distribution.png  (99.4 KB)
  figures\class_mean_images.png  (56.4 KB)
  figures\class_similarity_heatmap.png  (199.4 KB)
  figures\image_brightness_distribution.png  (76.4 KB)
  figures\pixel_histogram.png  (78.1 KB)
  figures\sample_images.png  (342.1 KB)
  figures\spatial_by_class.png  (138.8 KB)
  figures\spatial_heatmaps.png  (68.0 KB)
  preprocessing_config.json  (1.3 KB)
  spatial_maps.npz  (11.4 KB)
  split_indices.npz  (89.6 KB)
  split_metadata.json  (0.8 KB)
  tables\class_correlation_matrix.csv  (1.9 KB)
  tables\class_distribution.csv  (0.6 KB)
  tables\class_similarity_matrix.csv  (2.0 KB)
  tables\data_cleaning_log.csv  (0.2 KB)
  tables\data_review.csv  (0.4 KB)
  tables\eda_to_fe_handoff.csv  (2.1 KB)
  tables\integ

## E11. Gioi han cua phan EDA nay

Ghi ro de bao cao khong phong dai pham vi da lam:

1. **Khong co phan tich outlier dinh luong.** Cac anh "bat thuong" chi duoc phat hien qua
   kiem tra tich hop (toan den, trung lap) va qua quan sat dinh tinh o luoi mau. Khong co
   diem outlier theo PCA reconstruction hay kNN.
2. **Khong co near-duplicate detection.** Tim near-duplicate tren 60,000 anh la O(n^2) neu
   khong co index; nam ngoai pham vi lab. Chi exact duplicate duoc do.
3. **Mean image lam mat phuong sai trong lop.** Moi nhan xet tu Step 8 chi la gia thuyet.
4. **Khong co kiem dinh thong ke drift.** Train va validation cung lay tu mot pool; khac biet
   quan sat duoc la sampling noise, khong phai drift.
5. **Tap test chua duoc phan tich chi tiet.** Dung theo spec: chi mo sau khi khoa cau hinh.
6. **Spatial features phu thuoc threshold.** `FOREGROUND_THRESHOLD = 0.1` la lua chon co ly do
   nhung khong duy nhat; cac ket luan ve bounding box nen duoc doc kem gia tri nay.